Installing odfpy

In [ ]:
!pip install odfpy

In [ ]:
from odf.opendocument import load
from odf.table import Table, TableRow, TableCell
from odf.text import P
from odf import teletype
import pandas as pd

df = pd.read_csv('/content/Ride_Hailing_Apps_Survey_Pakistan.csv')
df.to_excel('ride_haling_apps.ods', engine='odf', index=False)

file_name = 'ride_haling_apps.ods'
doc = load(file_name)

print(f'The file to be analyze is: {file_name}')

The file to be analyze is: ride_haling_apps.ods


In [ ]:
from odf.opendocument import OpenDocumentSpreadsheet

sheets = doc.getElementsByType(Table)
sheet = sheets[0]
table = sheet.getElementsByType(Table)
rows = sheet.getElementsByType(TableRow)

# Displaying headers
header_row = rows[0]
headers = [
    teletype.extractText(cell.getElementsByType(P)[0]) if cell.getElementsByType(P) else ''
    for cell in header_row.getElementsByType(TableCell)
]
print(f'Headers: {headers}')

Headers: ['timestamp', 'age', 'gender', 'occupation', 'how_often_do_you_use_ride-hailing_apps?', 'which_ride-hailing_apps_have_you_used_so_far?', 'how_would_you_rate_the_pricing_features_of_ride-hailing_services?', 'rate_the_drive_behavior_of_ride-hailing_apps?', 'rate_the_app_usability_of_ride-hailing_services?', 'how_would_you_rate_the_safety_of_ride-hailing_services?', 'challenges_you_faced_with_ride-hailing_services?', 'what_measures_you_took_to_overcome_this_challenge?', 'which_service_you_consider_the_best_option?', 'why_do_you_consider_this_service_as_best_option?', 'do_you_prefer_using_online_apps_over_physical?_if_yes,_why?', 'column_16', 'consistency_flag', 'adjusted_flag']


In [ ]:
# Displaying sample rows
sample_rows = []
for i, row in enumerate(rows[1:6], 1):
  cells = row.getElementsByType(TableCell)
  value = [
      teletype.extractText(cell.getElementsByType(P)[0]) if cell.getElementsByType(P) else ''
      for cell in cells
  ]
  sample_rows.append(value)
  print(f'{i}: {value}')

1: ['', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', 'Valid', 'Original']
2: ['46:04.7', '25-34', 'Male', 'Employed', 'Weekly', 'Yango, InDrive', '7.0', '5.0', '8.0', '6.0', 'Unprofessional drivers', 'Reported the issue', 'InDrive', 'Usability', 'Affordability, Accessibility', '', 'Valid', 'Original']
3: ['47:06.8', '18-24', 'Male', 'Student', 'Monthly', 'InDrive', '8.0', '9.0', '6.0', '7.0', 'Long wait times', 'Switched to another app', 'InDrive', 'Usability', 'Affordability, Accessibility', '', 'Valid', 'Original']
4: ['00:37.3', '18-24', 'Female', 'Student', 'Weekly', 'Yango, InDrive, Careem, Uber', '6.0', '8.0', '10.0', '8.0', 'High prices, App issues', '', 'InDrive', 'Pricing, Security, Usability, Driver Behavior', 'Affordability, Accessibility, Driver Behavior, Safety', '', 'Valid', 'Original']
5: ['03:23.4', '25-34', 'Female', 'Housewife', 'Monthly', 'InDrive, Uber', '7.0', '9.0', '8.0', '9.0', 'High prices', 'Reported the issue', 'Uber', 'Security', 'Accessibilit

In [ ]:
# data cleaning
# create new file, doc and table
clean_file = 'CleanRideHailingAppsSurveyAnalysis.ods'
clean_doc = OpenDocumentSpreadsheet()
clean_table = Table()

# replace long names with short ones
header_mapping = {
    'timestamp': 'timestamp',
    'age': 'age',
    'gender': 'gender',
    'occupation': 'occupation',
    'how_often_do_you_use_ride-hailing_apps?': 'usage_frequency',
    'which_ride-hailing_apps_have_you_used_so_far?': 'apps_used',
    'how_would_you_rate_the_pricing_features_of_ride-hailing_services?': 'rating_pricing',
    'rate_the_drive_behavior_of_ride-hailing_apps?': 'rating_driver_behavior',
    'rate_the_app_usability_of_ride-hailing_services?': 'rating_usability',
    'how_would_you_rate_the_safety_of_ride-hailing_services?': 'rating_safety',
    'challenges_you_faced_with_ride-hailing_services?': 'challenges_faced',
    'what_measures_you_took_to_overcome_this_challenge?': 'user_mitigation',
    'which_service_you_consider_the_best_option?': 'preferred_app',
    'why_do_you_consider_this_service_as_best_option?': 'preferred_app_reason',
    'do_you_prefer_using_online_apps_over_physical?_if_yes,_why?': 'digital_preference_reason',
    'column_16': 'column_16',
    'consistency_flag': 'consistency_flag',
    'adjusted_flag': 'adjusted_flag'
}
# headers data cleaning
clean_headers = []
for h in headers:
  clean_values = h.lower().strip().replace(' ', '_') if h else ''
  short_names = header_mapping.get(clean_values, clean_values)
  clean_headers.append(short_names)

# rows data cleaning
clean_data = []

for row in rows[1:]:
  cells = row.getElementsByType(TableCell)
  values = []
  for cell in cells:
      ps = cell.getElementsByType(P)[0] if cell.getElementsByType(P) else ''
      if ps:
        text_val = teletype.extractText(ps).strip().lower()
      else:
        text_val = ''
      values.append(text_val)

# handling unequal length of columns
  if len(clean_headers) > len(values):
    values += [''] * (len(clean_headers) - len(values))
  values = [v if v != '' else None for v in values]
  record = dict(zip(clean_headers, values))
  clean_data.append(record)

print(f"Loaded and cleaned {len(clean_data)} records.")
print("Sample cleaned record:")
print(clean_data[3])


Loaded and cleaned 131 records.
Sample cleaned record:
{'timestamp': '00:37.3', 'age': '18-24', 'gender': 'female', 'occupation': 'student', 'usage_frequency': 'weekly', 'apps_used': 'yango, indrive, careem, uber', 'rating_pricing': '6.0', 'rating_driver_behavior': '8.0', 'rating_usability': '10.0', 'rating_safety': '8.0', 'challenges_faced': 'high prices, app issues', 'user_mitigation': None, 'preferred_app': 'indrive', 'preferred_app_reason': 'pricing, security, usability, driver behavior', 'digital_preference_reason': 'affordability, accessibility, driver behavior, safety', 'column_16': None, 'consistency_flag': 'valid', 'adjusted_flag': 'original'}


In [ ]:
# add clean data to data set
# adding headers
row = TableRow()
for h in clean_headers:
  cell = TableCell()
  cell.addElement(P(text=h))
  row.addElement(cell)
clean_table.addElement(row)
clean_doc.spreadsheet.addElement
# adding rows
